# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

#检查环境和API_KEY
# Load environment variables from .env
load_dotenv()

# Check and print results
# doublecheck_env("example.env")
doublecheck_env(".env")

OPENAI_API_KEY=****ywsA
DEEPSEEK_API_KEY=****c1e4
DEEPSEEK_BASE_URL=****.com
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_community.utilities import SQLDatabase
#把本地的 Chinook.db SQLite 数据库包装成一个 LangChain 可用的 SQLDatabase 对象。
#通过uri创建数据库对象
#sqlite:///表示是本地SQLite文件
#后面的Chinook.db就是数据库文件名
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
#最后将本地数据库文件打包装饰为能够通信的数据库对象

In [3]:
from dataclasses import dataclass
#这里的dataclass是一个官方的装饰器
#用来打包类,直接加上__init__初始化和__repr__打印等方法
@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

#打包为tool
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    #从上下文对象中获取数据库对象
    #不过为什么是context.db?
    #agent启动时会自动创建runtime,并且有context,store等属性
    #从下面可以看到,在agent启动时有context=RuntimeContext(db=db)
    #此时runtime的context就变为了RuntimeContext,其中有db数据库对象
    #而get_runtime函数中的RuntimeContext则只是声明runtime的context对象是RuntimeContext类型
    #于是整个流程就是:agent启动时注入上下文,执行工具时使用get_runtime函数获取上下文(并声明类型),获取这次运行的runtime后获取上下文中的db对象
    db = runtime.context.db

    try:
        #使用数据库对象尝试执行请求(query)
        return db.run(query)
    except Exception as e:
        #遇到错误就抛出
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT = """你是一名严谨的 SQLite 数据分析助手。

规则：
- 请逐步思考。
- 当你需要数据时，只能调用工具 `execute_sql`，并且一次只执行一条 SELECT 查询。
- 只允许只读操作；禁止使用 INSERT、UPDATE、DELETE、ALTER、DROP、CREATE、REPLACE、TRUNCATE。
- 除非用户明确要求，否则结果最多返回 5 行。
- 如果工具返回以 `Error:` 开头的错误信息，请修正 SQL 后再重试。
- 优先使用明确的列名，避免使用 SELECT *。
"""

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    # model="openai:gpt-5",
    model="deepseek-chat",
    #工具
    tools=[execute_sql],
    #提示词
    system_prompt=SYSTEM_PROMPT,
    #上下文schema,定义上下文结构
    context_schema=RuntimeContext,
)

## Repeated Queries

In [20]:
question = "This is Frank Harris, What was the total on my last invoice?response with Chinese."
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    #这里的列表是为了出现错误时方便查找错误,在案例中没有实际用途
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?response with Chinese.
================================== Ai Message ==================================

我来帮您查询Frank Harris的最后一张发票的总金额。首先，我需要找到Frank Harris的客户信息，然后查询他的最后一张发票。
Tool Calls:
  execute_sql (call_00_sArl2bRhTjiSPjdJ3JTE4D2l)
 Call ID: call_00_sArl2bRhTjiSPjdJ3JTE4D2l
  Args:
    query: SELECT CustomerId, FirstName, LastName FROM customers WHERE FirstName = 'Frank' AND LastName = 'Harris'
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: customers
[SQL: SELECT CustomerId, FirstName, LastName FROM customers WHERE FirstName = 'Frank' AND LastName = 'Harris']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================

让我先查看数据库中有哪些表：
Tool Calls:
  execute_sql (c

In [10]:
question = "What were the titles?response with Chinese."

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

我需要先了解您询问的是哪个表格中的标题信息。您能提供更多上下文吗？比如：

1. 您指的是哪个数据库中的哪个表格？
2. 您想查询什么类型的标题？比如文章标题、电影标题、书籍标题等？

或者，我可以先查看一下数据库中有哪些表格，然后您再告诉我具体想查询哪个表格的标题信息。
Tool Calls:
  execute_sql (call_00_JkODkwEMnlQ053sQJXk8aLBW)
 Call ID: call_00_JkODkwEMnlQ053sQJXk8aLBW
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================

我看到数据库中有多个表格。让我查看一下这些表格的结构，以便了解哪个表格包含标题信息：
Tool Calls:
  execute_sql (call_00_AN58g3AWZcXU4PFZIcM1sbvN)
 Call ID: call_00_AN58g3AWZcXU4PFZIcM1sbv

## Add memory

In [11]:
from langgraph.checkpoint.memory import InMemorySaver

In [12]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    # model="openai:gpt-5",
    model="deepseek-chat",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    #增加记忆配置,记忆会保存到python进程中
    checkpointer=InMemorySaver(),
)

In [13]:
question = "This is Frank Harris, What was the total on my last invoice?response with Chinese."
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    #指定同一个对话线程id,这样记忆才能落在同一条会话里
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?response with Chinese.
================================== Ai Message ==================================

我来帮您查询Frank Harris的最后一张发票的总金额。首先，我需要找到Frank Harris的客户ID，然后查询他的最后一张发票。

让我先查看数据库中有哪些表：
Tool Calls:
  execute_sql (call_00_r6qGboim6gQ5GF3t9VwDYBZz)
 Call ID: call_00_r6qGboim6gQ5GF3t9VwDYBZz
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================

现在我需要找到Frank Harris的客户ID：
Tool Calls:
  execute_sql (call_00_Sva9nSWPohitbLyPmjX5c05b)
 Call ID: call_00_Sva9nSWPohitbLyPmjX5c05b
  Arg

In [14]:
question = "What were the titles?并回答上一个问题是什么?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    #继续使用同一个对话线程id,这样更直观地看到记忆是否生效
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?并回答上一个问题是什么?
================================== Ai Message ==================================

让我先回答您的问题顺序：

1. **上一个问题是什么？**
   上一个问题是："This is Frank Harris, What was the total on my last invoice?"（这是Frank Harris，我最后一张发票的总金额是多少？）

2. **现在回答您的新问题："What were the titles?"（有哪些标题？）**

我需要先理解您指的是什么标题。根据上下文，您可能指的是Frank Harris最后一张发票中包含的曲目标题。让我查询发票ID 374中的具体项目：
Tool Calls:
  execute_sql (call_00_m1MncV616WJ35rRiSnunCuFl)
 Call ID: call_00_m1MncV616WJ35rRiSnunCuFl
  Args:
    query: SELECT il.InvoiceLineId, t.Name AS TrackTitle, il.UnitPrice, il.Quantity FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId WHERE il.InvoiceId = 374 ORDER BY il.InvoiceLineId
================================= Tool Message =================================
Name: execute_sql

[(2021, 'Holier Than Thou', 0.99, 1), (2022, 'Through The Never', 0.99, 1), (2023, 'My Friend Of Misery', 0.99, 1), (2024, 'The Wait', 

## Try your own queries
Now that there is memory, check the agents recall!

In [ ]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

In [10]:
from dotenv import load_dotenv
load_dotenv()
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional
from pydantic import BaseModel, Field, PositiveInt
from langchain_core.tools import tool


In [11]:
class MonthlyPaymentInput(BaseModel):
    """月供计算工具的输入。"""

    loan_amount: Decimal = Field(gt=0, description="贷款本金，单位：元")
    annual_interest_rate: Decimal = Field(gt=0, description="年利率，小数形式，例如 0.032")
    loan_years: PositiveInt = Field(description="贷款年限，单位：年")

def _calculate_monthly_payment(
        loan_amount: Decimal,
        annual_interest_rate: Decimal,
        loan_years: int,
)->dict:
    #转换为月
    months=loan_years*12
    #转换为月利率
    monthly_rate=annual_interest_rate/12
    #由于输入时规定了利率大于0,因此不考虑利率为0的情况
    #接下来就是按照公式计算月还款额=贷款数*月利率*(1+月利率)^月数/((1+月利率)^月数-1)
    #先计算复利因子(1+月利率)^月数
    x=(Decimal("1")+monthly_rate)**months
    monthly_payment=loan_amount*monthly_rate*x/(x-1)
    #最后返回字典包含月还款额,总还款额以及总利息
    #总偿还金额
    total_payment=monthly_payment*months
    #总利息
    total_interest=monthly_payment*months-loan_amount
    return{
        #这里使用quantize来限制小数位数,并且使用的是四舍五入
        "monthly_payment": monthly_payment.quantize(Decimal("0.01"),rounding=ROUND_HALF_UP),
        "total_payment": total_payment.quantize(Decimal("0.01"),rounding=ROUND_HALF_UP),
        "total_interest": total_interest.quantize(Decimal("0.01"),rounding=ROUND_HALF_UP),
    }
#进行测试结果应该是下面的
# {'monthly_payment': Decimal('2823.31'),
#  'total_payment': Decimal('677595.40'),
#  'total_interest': Decimal('177595.40')}
# _calculate_monthly_payment(
#     loan_amount=Decimal("500000"),
#     annual_interest_rate=Decimal("0.032"),
#     loan_years=20,
# )

#函数和约束都写好了接下来写工具
@tool(args_schema=MonthlyPaymentInput)
def calculate_monthly_payment(
        loan_amount: Decimal,
        annual_interest_rate: Decimal,
        loan_years: int,
)->str:
    """计算月供,总还款额以及总利息"""
    result=_calculate_monthly_payment(loan_amount, annual_interest_rate, loan_years)

    return (f"月供为{result['monthly_payment']}元;"
            f"总还款额为 {result['total_payment']} 元；"
            f"总利息为 {result['total_interest']} 元。"
            )

In [12]:
LOAN_ADVISOR_PROMPT = """
你是一名贷款顾问助手。

你的职责：
1. 结合同一线程中的历史对话，记住用户已经提供的信息，例如姓名、月收入、首付、贷款年限、目标月供。
2. 如果用户后续追问，优先复用之前已经给出的信息，不要反复要求用户重复输入。
3. 如果计算月供所需的信息不完整，先明确指出缺少哪些字段，再继续追问。
4. 只要涉及月供、总还款额、总利息的精确计算，必须调用工具，不要自行估算。
5. 回答默认使用中文，语气简洁清楚。
"""

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent=create_agent(
    model="deepseek-chat",
    system_prompt=LOAN_ADVISOR_PROMPT,
    checkpointer=InMemorySaver(),
    tools=[calculate_monthly_payment],
)


In [15]:
for step,metadata in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "你好,我想咨询一下贷款相关的问题"
            }
        ]
    },
    {
        "configurable": {
            "thread_id": "1"
        }
    },
    stream_mode="messages",
):
    print(step.content,end="")

questions = [
    "我要贷50万",
    "利率是0.032",
    "贷款年限是30年"
]
for _ in range(3):
    print()
    for step,metadata in agent.stream(
            {"messages": [{"role": "user","content": questions[_],}]},
            {"configurable": {"thread_id": "1"}},
        stream_mode="messages",
    ):
        print(step.content,end="")



您好！我是贷款顾问助手。

我看到您之前咨询过贷款50万元、年利率3.2%、贷款30年的情况，计算结果如下：
- 月供：2,162.33元
- 总还款额：778,440.36元  
- 总利息：278,440.36元

您是想要咨询新的贷款方案，还是对之前的计算结果有疑问呢？
好的，您需要贷款50万元。

根据之前的对话记录，您之前咨询过50万元贷款，当时使用的是：
- 年利率：0.032（3.2%）
- 贷款年限：30年

请问这次您是想：
1. 使用相同的利率和年限（3.2%，30年）重新计算？
2. 还是想使用不同的利率或年限？

请告诉我您的选择，或者提供新的利率和年限信息。
好的，现在我知道：
- 贷款金额：50万元
- 年利率：0.032（3.2%）

还差贷款年限信息。您之前咨询时使用的是30年，这次您想用多少年的贷款年限呢？

请告诉我贷款年限，我就可以为您计算详细的月供、总还款额和总利息了。
好的，现在所有信息都齐全了：
- 贷款金额：50万元
- 年利率：0.032（3.2%）
- 贷款年限：30年

我为您计算一下详细的还款情况：月供为2162.33元;总还款额为 778440.36 元；总利息为 278440.36 元。根据您提供的信息，计算结果如下：

**贷款详情：**
- 贷款金额：500,000元
- 年利率：3.2%
- 贷款年限：30年

**还款计划：**
- **月供**：2,162.33元
- **总还款额**：778,440.36元
- **总利息**：278,440.36元

这与您之前咨询的结果完全一致。您每月需要还款约2,162元，30年总共需要还款约77.8万元，其中利息部分约为27.8万元。

请问您还有其他问题吗？